# 07 — Gold Layer: Data Marts (Descriptivo + Diagnóstico)

Lee desde `tlc_silver` y materializa **6 Data Marts** en `tlc_gold`.

### Marts Descriptivos
- `mart_demand_volume` — Volumen de viajes por fecha/hora/zona/vehículo
- `mart_financial_performance` — Ingresos, propinas y peajes
- `mart_operational_profile` — Velocidades y tiempos de viaje

### Marts Diagnósticos
- `mart_congestion_impact` — Impacto del impuesto de congestión CBD
- `mart_abc_xyz_zones` — Clasificación Pareto ABC + estabilidad XYZ por zona
- `mart_supply_demand_balance` — Balance oferta/demanda por zona

> **Arquitectura de Memoria:** Cada mart se procesa de forma independiente (leer → agregar → escribir → liberar).
> Esto evita cargar los 700M+ registros en memoria de golpe.
> El resultado final pesa <50 MB en MongoDB independientemente del volumen de Silver.

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from core.config.settings import settings
import pyspark.sql.functions as F
from functools import reduce

print('Imports OK.')

Imports OK.


In [2]:
spark = get_spark('TLC_Gold_Marts')

# Reducir particiones de shuffle: con datos ya agregados no necesitamos 2000
spark.conf.set('spark.sql.shuffle.partitions', '200')

print(f'Spark {spark.version} ready.')
print(f'Shuffle partitions: {spark.conf.get("spark.sql.shuffle.partitions")}')

2026-07-19 09:41:03 | INFO     | tlc.spark_utils | [SPARK] Session ready | version=3.4.1 master=local[*]
Spark 3.4.1 ready.
Shuffle partitions: 200


In [3]:
# ── Cargar dimensión de zonas una sola vez (265 filas, trivial en memoria)
dim_zone = read_mongo(spark, settings.MONGO_DB_GOLD, 'dim_zone')
dim_zone.cache()  # 265 filas — siempre cabe en RAM
print(f'dim_zone cargada: {dim_zone.count()} zonas')

2026-07-19 09:41:04 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_gold.dim_zone
dim_zone cargada: 265 zonas


In [4]:
# ── Helper: leer Silver proyectando SOLO las columnas necesarias
# CLAVE: al proyectar antes de cualquier transformación, MongoDB solo
# envía por red las columnas que realmente vamos a usar, reduciendo el
# volumen de datos transferidos desde Mongo a Spark a una fracción mínima.

SILVER_COLLECTIONS = ['trips_yellow', 'trips_green', 'trips_fhv', 'trips_hvfhv']

def read_silver(columns: list) -> 'DataFrame':
    """
    Lee y une todas las colecciones Silver proyectando solo las columnas indicadas.
    Devuelve un DataFrame con las columnas solicitadas más vehicle_type y pickup_dt.
    """
    base_select = [
        F.col('_meta.vehicle_type').alias('vehicle_type'),
        F.col('datetimes.pickup').alias('pickup_dt'),
    ] + columns

    dfs = []
    for coll in SILVER_COLLECTIONS:
        try:
            df = read_mongo(spark, settings.MONGO_DB_SILVER, coll).select(*base_select)
            dfs.append(df)
        except Exception as e:
            print(f'  ✗ {coll}: {e}')

    if not dfs:
        raise RuntimeError('No se encontraron colecciones Silver.')

    unified = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

    # Columnas de tiempo derivadas — se calculan UNA VEZ y solo sobre lo necesario
    return (
        unified
        .withColumn('year',        F.year('pickup_dt'))
        .withColumn('month',       F.month('pickup_dt'))
        .withColumn('day',         F.dayofmonth('pickup_dt'))
        .withColumn('hour',        F.hour('pickup_dt'))
        .withColumn('pickup_date', F.to_date('pickup_dt'))
        .withColumn('is_weekend',
            F.when(F.dayofweek('pickup_dt').isin(1, 7), True).otherwise(False)
        )
    )

print('Helper read_silver() definido.')

Helper read_silver() definido.


---
## Mart 1: `mart_demand_volume` — Volumen de Viajes
**Dashboard Descriptivo 1:** Mapa de calor y curvas de demanda por hora, zona y tipo de vehículo.

In [5]:
print('Procesando mart_demand_volume...')

df = read_silver([
    F.col('locations.pickup.zone_id').alias('pickup_zone_id'),
    F.coalesce(F.col('datetimes.duration_minutes'), F.lit(0.0)).alias('duration_minutes'),
    F.coalesce(F.col('metrics.distance_miles'),   F.lit(0.0)).alias('distance_miles'),
    F.coalesce(F.col('metrics.passenger_count'),  F.lit(0)).alias('passenger_count'),
])

mart = (
    df
    .groupBy('vehicle_type', 'year', 'month', 'day', 'hour', 'is_weekend', 'pickup_zone_id')
    .agg(
        F.count('*').alias('total_trips'),
        F.sum('passenger_count').alias('total_passengers'),
        F.avg('duration_minutes').alias('avg_duration_min'),
        F.avg('distance_miles').alias('avg_distance_miles'),
    )
    .join(
        F.broadcast(dim_zone).select(F.col('zone_id'), F.col('zone_name'), F.col('borough')),
        F.col('pickup_zone_id') == F.col('zone_id'), 'left'
    )
    .drop('zone_id')
)

write_mongo(mart, settings.MONGO_DB_GOLD, 'mart_demand_volume', mode='overwrite')
del df, mart  # liberar referencia antes del siguiente mart
print('✓ mart_demand_volume escrito')

Procesando mart_demand_volume...
2026-07-19 09:41:06 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_yellow
2026-07-19 09:41:07 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_green
2026-07-19 09:41:08 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_fhv
2026-07-19 09:41:10 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_hvfhv
2026-07-19 10:16:46 | INFO     | tlc.spark_utils | [SPARK] Wrote Unknown rows → MongoDB tlc_gold.mart_demand_volume (mode=overwrite)
✓ mart_demand_volume escrito


---
## Mart 2: `mart_financial_performance` — Rendimiento Financiero
**Dashboard Descriptivo 2:** Ingresos totales, propinas, peajes y distribución de métodos de pago.

In [6]:
print('Procesando mart_financial_performance...')

df = read_silver([
    F.col('locations.pickup.zone_id').alias('pickup_zone_id'),
    F.coalesce(F.col('financials.total_amount'),          F.lit(0.0)).alias('total_amount'),
    F.coalesce(F.col('financials.fare_amount'),           F.lit(0.0)).alias('fare_amount'),
    F.coalesce(F.col('financials.tip_amount'),            F.lit(0.0)).alias('tip_amount'),
    F.coalesce(F.col('financials.tolls_amount'),          F.lit(0.0)).alias('tolls_amount'),
    F.coalesce(F.col('financials.congestion_surcharge'),  F.lit(0.0)).alias('congestion_surcharge'),
    F.coalesce(F.col('financials.cbd_congestion_fee'),    F.lit(0.0)).alias('cbd_congestion_fee'),
    F.coalesce(F.col('financials.payment_type'),          F.lit('Unknown')).alias('payment_type'),
])

mart = (
    df
    .groupBy('vehicle_type', 'year', 'month', 'pickup_zone_id', 'payment_type')
    .agg(
        F.count('*').alias('total_trips'),
        F.sum('total_amount').alias('total_revenue'),
        F.avg('total_amount').alias('avg_revenue_per_trip'),
        F.sum('fare_amount').alias('total_fare'),
        F.sum('tip_amount').alias('total_tips'),
        F.avg('tip_amount').alias('avg_tip'),
        F.sum('tolls_amount').alias('total_tolls'),
        F.sum('congestion_surcharge').alias('total_congestion_surcharge'),
        F.sum('cbd_congestion_fee').alias('total_cbd_fee'),
        F.avg(
            F.when(F.col('tip_amount') > 0, 1).otherwise(0)
        ).alias('tip_rate_pct'),
    )
    .join(
        F.broadcast(dim_zone).select(F.col('zone_id'), F.col('zone_name'), F.col('borough')),
        F.col('pickup_zone_id') == F.col('zone_id'), 'left'
    )
    .drop('zone_id')
)

write_mongo(mart, settings.MONGO_DB_GOLD, 'mart_financial_performance', mode='overwrite')
del df, mart
print('✓ mart_financial_performance escrito')

Procesando mart_financial_performance...
2026-07-19 10:16:48 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_yellow
2026-07-19 10:16:49 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_green
2026-07-19 10:16:50 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_fhv
2026-07-19 10:16:52 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_hvfhv
2026-07-19 11:01:42 | INFO     | tlc.spark_utils | [SPARK] Wrote Unknown rows → MongoDB tlc_gold.mart_financial_performance (mode=overwrite)
✓ mart_financial_performance escrito


---
## Mart 3: `mart_operational_profile` — Perfil Operativo
**Dashboard Descriptivo 3:** Velocidades promedio, tiempos de viaje y distancias por corredor.

In [7]:
print('Procesando mart_operational_profile...')

df = read_silver([
    F.col('locations.pickup.zone_id').alias('pickup_zone_id'),
    F.col('locations.dropoff.zone_id').alias('dropoff_zone_id'),
    F.coalesce(F.col('datetimes.duration_minutes'), F.lit(0.0)).alias('duration_minutes'),
    F.coalesce(F.col('metrics.distance_miles'),   F.lit(0.0)).alias('distance_miles'),
])

dim_pu = dim_zone.select(
    F.col('zone_id').alias('pu_zone_id'),
    F.col('zone_name').alias('pickup_zone_name'),
    F.col('borough').alias('pickup_borough'),
)
dim_do = dim_zone.select(
    F.col('zone_id').alias('do_zone_id'),
    F.col('zone_name').alias('dropoff_zone_name'),
    F.col('borough').alias('dropoff_borough'),
)

mart = (
    df
    .withColumn(
        'speed_mph',
        F.when(
            (F.col('duration_minutes') > 1) & (F.col('distance_miles') > 0),
            F.col('distance_miles') / (F.col('duration_minutes') / 60.0)
        ).otherwise(None)
    )
    .groupBy('vehicle_type', 'year', 'month', 'hour', 'pickup_zone_id', 'dropoff_zone_id')
    .agg(
        F.count('*').alias('total_trips'),
        F.avg('duration_minutes').alias('avg_duration_min'),
        F.avg('distance_miles').alias('avg_distance_miles'),
        F.avg('speed_mph').alias('avg_speed_mph'),
        F.percentile_approx('duration_minutes', 0.5).alias('median_duration_min'),
        F.percentile_approx('distance_miles', 0.5).alias('median_distance_miles'),
    )
    .join(F.broadcast(dim_pu), F.col('pickup_zone_id') == F.col('pu_zone_id'), 'left').drop('pu_zone_id')
    .join(F.broadcast(dim_do), F.col('dropoff_zone_id') == F.col('do_zone_id'), 'left').drop('do_zone_id')
)

write_mongo(mart, settings.MONGO_DB_GOLD, 'mart_operational_profile', mode='overwrite')
del df, mart
print('✓ mart_operational_profile escrito')

Procesando mart_operational_profile...
2026-07-19 11:01:45 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_yellow
2026-07-19 11:01:46 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_green
2026-07-19 11:01:47 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_fhv
2026-07-19 11:01:50 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_hvfhv
2026-07-19 11:58:16 | INFO     | tlc.spark_utils | [SPARK] Wrote Unknown rows → MongoDB tlc_gold.mart_operational_profile (mode=overwrite)
✓ mart_operational_profile escrito


---
## Mart 4: `mart_congestion_impact` — Impacto del Impuesto de Congestión
**Dashboard Diagnóstico 4:** Correlación entre el cobro del CBD ($2.50) y comportamiento de propinas y pagos.

In [ ]:
print('Procesando mart_congestion_impact...')

df = read_silver([
    F.col('locations.pickup.zone_id').alias('pickup_zone_id'),
    F.coalesce(F.col('financials.tip_amount'),         F.lit(0.0)).alias('tip_amount'),
    F.coalesce(F.col('financials.fare_amount'),        F.lit(0.0)).alias('fare_amount'),
    F.coalesce(F.col('financials.total_amount'),       F.lit(0.0)).alias('total_amount'),
    F.coalesce(F.col('financials.cbd_congestion_fee'), F.lit(0.0)).alias('cbd_congestion_fee'),
    F.coalesce(F.col('financials.congestion_surcharge'), F.lit(0.0)).alias('congestion_surcharge'),
    F.coalesce(F.col('financials.payment_type'),       F.lit('Unknown')).alias('payment_type'),
])

mart = (
    df
    .withColumn('has_cbd_fee',
        F.when(F.col('cbd_congestion_fee') > 0, True).otherwise(False))
    .withColumn('has_congestion_surcharge',
        F.when(F.col('congestion_surcharge') > 0, True).otherwise(False))
    .groupBy('vehicle_type', 'year', 'month', 'pickup_zone_id', 'payment_type',
             'has_cbd_fee', 'has_congestion_surcharge')
    .agg(
        F.count('*').alias('total_trips'),
        F.avg('tip_amount').alias('avg_tip'),
        F.avg('total_amount').alias('avg_total'),
        F.sum('cbd_congestion_fee').alias('total_cbd_collected'),
        F.sum('congestion_surcharge').alias('total_surcharge_collected'),
        F.avg(
            F.when(F.col('fare_amount') > 0, F.col('tip_amount') / F.col('fare_amount'))
             .otherwise(0.0)
        ).alias('tip_to_fare_ratio'),
    )
    .join(
        F.broadcast(dim_zone).select(F.col('zone_id'), F.col('zone_name'), F.col('borough')),
        F.col('pickup_zone_id') == F.col('zone_id'), 'left'
    )
    .drop('zone_id')
)

write_mongo(mart, settings.MONGO_DB_GOLD, 'mart_congestion_impact', mode='overwrite')
del df, mart
print('✓ mart_congestion_impact escrito')

Procesando mart_congestion_impact...
2026-07-19 11:58:21 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_yellow
2026-07-19 11:58:23 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_green
2026-07-19 11:58:25 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_fhv
2026-07-19 11:58:27 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_silver.trips_hvfhv


---
## Mart 5: `mart_abc_xyz_zones` — Clasificación Pareto ABC + Estabilidad XYZ
**Dashboard Diagnóstico 5:** Identifica las zonas Clase A (top 80% de ingresos) y su estabilidad temporal.

- **ABC:** A = 80% del ingreso acumulado | B = siguiente 15% | C = resto
- **XYZ:** X = CV < 0.5 (estable) | Y = 0.5–1.0 | Z = CV > 1.0 (volátil)

In [ ]:
import pyspark.sql.window as W

print('Procesando mart_abc_xyz_zones...')

df = read_silver([
    F.col('locations.pickup.zone_id').alias('pickup_zone_id'),
    F.coalesce(F.col('financials.total_amount'), F.lit(0.0)).alias('total_amount'),
])

# Paso 1: Ingreso y volumen mensual por zona
zone_monthly = (
    df
    .groupBy('pickup_zone_id', 'year', 'month')
    .agg(
        F.sum('total_amount').alias('monthly_revenue'),
        F.count('*').alias('monthly_trips'),
    )
)

# Paso 2: Totales y estadísticas de variabilidad por zona
zone_stats = (
    zone_monthly
    .groupBy('pickup_zone_id')
    .agg(
        F.sum('monthly_revenue').alias('total_revenue'),
        F.sum('monthly_trips').alias('total_trips'),
        F.avg('monthly_trips').alias('avg_monthly_trips'),
        F.stddev('monthly_trips').alias('stddev_monthly_trips'),
    )
    .withColumn(
        'cv',
        F.when(
            F.col('avg_monthly_trips') > 0,
            F.col('stddev_monthly_trips') / F.col('avg_monthly_trips')
        ).otherwise(0.0)
    )
)

# Paso 3: Total global para Pareto — collect() es seguro aquí (1 fila)
grand_total = zone_stats.agg(F.sum('total_revenue')).collect()[0][0] or 1.0

win_revenue = W.Window.orderBy(F.col('total_revenue').desc())
zone_classified = (
    zone_stats
    .withColumn('revenue_rank', F.rank().over(win_revenue))
    .withColumn('cumulative_revenue', F.sum('total_revenue').over(win_revenue))
    .withColumn('cumulative_pct', F.col('cumulative_revenue') / F.lit(grand_total))
    .withColumn(
        'abc_class',
        F.when(F.col('cumulative_pct') <= 0.80, 'A')
         .when(F.col('cumulative_pct') <= 0.95, 'B')
         .otherwise('C')
    )
    .withColumn(
        'xyz_class',
        F.when(F.col('cv') < 0.5, 'X')
         .when(F.col('cv') < 1.0, 'Y')
         .otherwise('Z')
    )
    .withColumn('segment', F.concat(F.col('abc_class'), F.col('xyz_class')))
)

mart = (
    zone_classified
    .join(
        F.broadcast(dim_zone).select(
            F.col('zone_id'), F.col('zone_name'), F.col('borough'), F.col('service_zone')
        ),
        F.col('pickup_zone_id') == F.col('zone_id'), 'left'
    )
    .drop('zone_id')
)

write_mongo(mart, settings.MONGO_DB_GOLD, 'mart_abc_xyz_zones', mode='overwrite')
del df, zone_monthly, zone_stats, zone_classified, mart
print('✓ mart_abc_xyz_zones escrito')

---
## Mart 6: `mart_supply_demand_balance` — Balance Oferta/Demanda por Zona
**Dashboard Diagnóstico 6:** Detecta zonas 'muertas' donde los taxis llegan vacíos (Dropoff alto, Pickup bajo).

In [ ]:
print('Procesando mart_supply_demand_balance...')

# Pickups (demanda): solo necesitamos zone_id de pickup
df_pu = read_silver([
    F.col('locations.pickup.zone_id').alias('zone_id'),
])
pickups = (
    df_pu
    .groupBy('zone_id', 'vehicle_type', 'year', 'month', 'hour')
    .agg(F.count('*').alias('pickups'))
)
del df_pu

# Dropoffs (oferta llegando): necesitamos zone_id de dropoff
# Leemos de nuevo con solo las columnas necesarias
SILVER_COLLECTIONS = ['trips_yellow', 'trips_green', 'trips_fhv', 'trips_hvfhv']
do_dfs = []
for coll in SILVER_COLLECTIONS:
    try:
        df_tmp = read_mongo(spark, settings.MONGO_DB_SILVER, coll).select(
            F.col('_meta.vehicle_type').alias('vehicle_type'),
            F.col('datetimes.pickup').alias('pickup_dt'),
            F.col('locations.dropoff.zone_id').alias('zone_id'),
        )
        do_dfs.append(df_tmp)
    except Exception as e:
        print(f'  ✗ {coll}: {e}')

df_do = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), do_dfs)
df_do = (
    df_do
    .withColumn('year',  F.year('pickup_dt'))
    .withColumn('month', F.month('pickup_dt'))
    .withColumn('hour',  F.hour('pickup_dt'))
)
dropoffs = (
    df_do
    .groupBy('zone_id', 'vehicle_type', 'year', 'month', 'hour')
    .agg(F.count('*').alias('dropoffs'))
)
del df_do, do_dfs

mart = (
    pickups
    .join(dropoffs, on=['zone_id', 'vehicle_type', 'year', 'month', 'hour'], how='outer')
    .fillna(0, subset=['pickups', 'dropoffs'])
    .withColumn('balance', F.col('pickups') - F.col('dropoffs'))
    .withColumn(
        'supply_status',
        F.when(F.col('balance') >  50, 'DEFICIT')
         .when(F.col('balance') < -50, 'SURPLUS')
         .otherwise('BALANCED')
    )
    .withColumn(
        'balance_ratio',
        F.when(F.col('dropoffs') > 0, F.col('pickups') / F.col('dropoffs')).otherwise(None)
    )
    .join(
        F.broadcast(dim_zone).select(F.col('zone_id'), F.col('zone_name'), F.col('borough')),
        on='zone_id', how='left'
    )
)

write_mongo(mart, settings.MONGO_DB_GOLD, 'mart_supply_demand_balance', mode='overwrite')
del pickups, dropoffs, mart
print('✓ mart_supply_demand_balance escrito')

In [ ]:
marts = [
    'mart_demand_volume',
    'mart_financial_performance',
    'mart_operational_profile',
    'mart_congestion_impact',
    'mart_abc_xyz_zones',
    'mart_supply_demand_balance',
]

print('=' * 55)
print('  CAPA GOLD — DATA MARTS ESCRITOS EXITOSAMENTE')
print('=' * 55)
for m in marts:
    print(f'  ✓ tlc_gold.{m}')
print('=' * 55)

dim_zone.unpersist()
print('Cache de dim_zone liberada.')
print('Siguiente paso: Ejecutar notebook 08_ml_sarima.ipynb')